In [9]:
"""
Арбитражная активность и LP-доходность — ETH/USDC 0.05% Uniswap v3
====================================================================
Подвыборка: 01.12.2025 – 28.02.2026 (N = 90 дней)
Y = log(1 + LP return daily)

Входные файлы (положить рядом со скриптом):
  1. основной датасет
  2.  арб. активность

Выходные файлы:
  dataset_90days.csv      — объединённый датасет 90 дней
  regression_results.csv  — коэффициенты всех моделей

Запуск:
  pip install pandas numpy scipy statsmodels
  python arbitrage_regression_v2.py
"""

import pandas as pd
import numpy as np
from scipy import stats as st
from statsmodels.stats.outliers_influence import variance_inflation_factor

SEP = "=" * 72

# ─── ПАРАМЕТРЫ ────────────────────────────────────────────────────────────────
MAIN_CSV = '/Users/aleksandra/Desktop/Дисертация/USDC - ETH 0.05 - uniswap_v3_ethusdc.csv'
ARB_CSV  = '/Users/aleksandra/Desktop/Дисертация/Arbitragers activity - USDC-ETH 0.05.csv'
OUT_DATA = "dataset_90days.csv"
OUT_REG  = "regression_results.csv"

# ─── 1. АРБИТРАЖНЫЕ ДАННЫЕ ────────────────────────────────────────────────────
print(SEP)
print("  ЗАГРУЗКА И АГРЕГАЦИЯ АРБИТРАЖНЫХ ДАННЫХ")
print(SEP)

raw = pd.read_csv(ARB_CSV, header=0, low_memory=False)
arb = raw.iloc[1:].copy()
arb.columns = raw.iloc[0].tolist()
arb["block_time"] = pd.to_datetime(arb["block_time"], errors="coerce")
arb = arb[arb["block_time"].notna()].sort_values("block_time").reset_index(drop=True)
arb["arb"]        = arb["arb_signal"].astype(str).str.upper() == "TRUE"
arb["spread_abs"] = pd.to_numeric(arb["Модуль спреда"], errors="coerce")
arb["spread_pct"] = pd.to_numeric(arb["spread_pct"],    errors="coerce")
arb["dex_vol"]    = pd.to_numeric(arb["dex_volume"],    errors="coerce")
arb["date"]       = arb["block_time"].dt.date

daily_arb = arb.groupby("date").agg(
    n_blocks   = ("block_time", "count"),
    n_arb      = ("arb",        "sum"),
    arb_rate   = ("arb",        "mean"),
    spread_abs = ("spread_abs", "mean"),
    spread_pct = ("spread_pct", "mean"),
    arb_vol    = ("dex_vol",    lambda x: x[arb.loc[x.index, "arb"]].sum()),
).reset_index()
daily_arb["date"] = pd.to_datetime(daily_arb["date"])

print(f"Блоков всего    : {len(arb):,}")
print(f"Дней            : {len(daily_arb)}")
print(f"Период          : {daily_arb['date'].min().date()} → {daily_arb['date'].max().date()}")
print(f"arb_rate  mean  : {daily_arb['arb_rate'].mean():.4f}  std={daily_arb['arb_rate'].std():.4f}")
print(f"spread_abs mean : {daily_arb['spread_abs'].mean():.4f}  std={daily_arb['spread_abs'].std():.4f}")

# ─── 2. ОСНОВНОЙ ДАТАСЕТ ─────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  ЗАГРУЗКА ОСНОВНОГО ДАТАСЕТА")
print(SEP)

raw2 = pd.read_csv(MAIN_CSV, skiprows=2, header=0)
col_names = [str(c).strip() for c in raw2.iloc[0].tolist()]
df = raw2.iloc[1:].copy()
df.columns = col_names
for c in df.columns:
    if c != "day":
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "").str.strip(),
                              errors="coerce")
df["day"] = pd.to_datetime(df["day"], errors="coerce")
df = df[df["day"].notna()].sort_values("day").reset_index(drop=True)
df_m = df[df["LP return daily"] != -1.0].copy()

print(f"Строк всего     : {len(df)}")
print(f"Рабочая выборка : {len(df_m)}")
print(f"Vol_ETH (новый) : mean={df_m['Vol_ETH'].mean():.6f}  "
      f"[rolling std внутридневных доходностей]")

# ─── 3. ПОСТРОЕНИЕ ПЕРЕМЕННЫХ ─────────────────────────────────────────────────
btc_open = next(c for c in df_m.columns if "BTC" in c and "open" in c.lower())

df_m = df_m.copy()
df_m["ETH_ret1"]     = df_m["ETHUSDC close"].pct_change(1)
df_m["ETH_ret7"]     = df_m["ETHUSDC close"].pct_change(7)
df_m["ETH_range"]    = ((df_m["ETHUSDC high"] - df_m["ETHUSDC low"])
                        / df_m["ETHUSDC open"].replace(0, np.nan))
df_m["ETH_ma30"]     = df_m["ETHUSDC close"].rolling(30, min_periods=10).mean()
df_m["BTC_range"]    = ((df_m["BTCUSDC high"] - df_m["BTCUSDC low"])
                        / df_m[btc_open].replace(0, np.nan))
df_m["log_Vol_Pool"] = np.log(df_m["Daily_Volume_Pool_ETH_USDC"].replace(0, np.nan))
df_m["log_TVL"]      = np.log(df_m["tvlUSD"].replace(0, np.nan))
df_m["log_txCount"]  = np.log(df_m["txCount"])
df_m["Vola_ETH_sq"]  = df_m["Vol_ETH"] ** 2
df_m["log_Vol_CEX"]  = np.log(df_m["ETHUSDC volume"].replace(0, np.nan))
df_m["bull"]         = ((df_m["ETHUSDC close"] > df_m["ETH_ma30"]) &
                        (df_m["ETH_ret7"] > 0.03)).astype(float)
df_m["bear"]         = ((df_m["ETHUSDC close"] < df_m["ETH_ma30"]) &
                        (df_m["ETH_ret7"] < -0.03)).astype(float)
df_m["log_LP_full"]  = np.log(1 + df_m["LP return daily"].clip(lower=-0.9999))
df_m["LP_lag1"]      = df_m["log_LP_full"].shift(1)
df_m["IL_pct"]       = (df_m["IL_yesterday"]
                        / df_m["Value of LP position"].replace(0, np.nan) * 100)
df_m["ETH_share"]    = ((df_m["ETH0"] * df_m["ETHUSDC close"])
                        / df_m["Value of LP position"].replace(0, np.nan))

# ─── 4. ОБЪЕДИНЕНИЕ ───────────────────────────────────────────────────────────
df_m["date"]      = df_m["day"].dt.normalize()
daily_arb["date"] = daily_arb["date"].dt.normalize()
merged = df_m.merge(daily_arb, on="date", how="inner")

print(f"\nОбъединённая выборка : N = {len(merged)}")
print(f"Период               : {merged['date'].min().date()} → {merged['date'].max().date()}")

# Корреляции
print("\nКорреляции с log_LP_full:")
for v in ["arb_rate", "spread_abs", "Vol_ETH", "ETH_ret1"]:
    print(f"  {v:<15} r = {merged['log_LP_full'].corr(merged[v]):+.4f}")

# ─── 5. OLS + HC3 ─────────────────────────────────────────────────────────────
def ols_hc3(X, y, names):
    n, kf = X.shape
    k  = kf - 1
    b  = np.linalg.lstsq(X, y, rcond=None)[0]
    yh = X @ b
    e  = y - yh
    ssr = np.sum(e ** 2)
    sst = np.sum((y - y.mean()) ** 2)
    r2  = 1 - ssr / sst
    r2a = 1 - (1 - r2) * (n - 1) / (n - k - 1)
    Xi  = np.linalg.pinv(X.T @ X)
    h   = np.einsum("ij,jk,ik->i", X, Xi, X)
    ea  = e / np.maximum(1 - h, 0.01)
    V   = Xi @ ((X * ea[:, None]).T @ (X * ea[:, None])) @ Xi
    se  = np.sqrt(np.clip(np.diag(V), 0, None))
    tv  = b / np.where(se > 0, se, 1e-10)
    pv  = 2 * st.t.sf(np.abs(tv), df=n - k - 1)
    fs  = ((sst - ssr) / k) / (ssr / (n - k - 1))
    fp  = st.f.sf(fs, k, n - k - 1)
    dw  = np.sum(np.diff(e) ** 2) / ssr
    jbs, jbp = st.jarque_bera(e)
    return dict(
        n=n, k=k, r2=r2, r2adj=r2a, f=fs, fp=fp, dw=dw,
        beta=b, se=se, t=tv, p=pv,
        jb=jbs, jbp=jbp, skew=st.skew(e), kurt=st.kurtosis(e),
        cols=names, resid=e,
    )


def run(xcols, ycol="log_LP_full", data=merged):
    use = [c for c in xcols if c in data.columns]
    s   = data[use + [ycol]].dropna().replace([np.inf, -np.inf], np.nan).dropna()
    X   = np.column_stack([np.ones(len(s))] + [s[c].values for c in use])
    return ols_hc3(X, s[ycol].values, ["const"] + use)


def sig(p):
    return "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.1 else ""))


def print_r(label, r, base_r2=None):
    delta = f"  Δ={r['r2adj']-base_r2:+.4f}" if base_r2 is not None else ""
    print(f"\n{SEP}")
    print(f"  {label}")
    print(f"  N={r['n']}  R²={r['r2']:.4f}  R²adj={r['r2adj']:.4f}{delta}")
    print(f"  F={r['f']:.2f} (p={r['fp']:.2e})  DW={r['dw']:.3f}  "
          f"JB={r['jb']:.1f} (p={r['jbp']:.4f})  Skew={r['skew']:.3f}  Kurt={r['kurt']:.3f}")
    print(f"  {'Переменная':<22} {'β':>11} {'SE_HC3':>9} {'t':>7} {'p':>9}  Знач.")
    print(f"  {'-'*65}")
    for i, nm in enumerate(r["cols"]):
        star = sig(r["p"][i])
        hi   = "  ←" if star and nm in ["arb_rate", "spread_abs"] else ""
        print(f"  {nm[:22]:<22} {r['beta'][i]:>11.5f} {r['se'][i]:>9.5f} "
              f"{r['t'][i]:>7.3f} {r['p'][i]:>9.4f}  {star:<3}{hi}")


# Базовые регрессоры C1
XC1 = ["log_Vol_Pool", "Vol_ETH", "Vola_ETH_sq", "ETH_range", "BTC_range",
        "Vol_BTC", "log_Vol_CEX", "log_TVL", "log_txCount",
        "ETH_ret1", "bull", "bear", "LP_lag1"]

print(f"\n{SEP}")
print("  OLS + HC3 | Y = log(1 + LP return daily)")
print(SEP)

r0 = run(XC1)
r1 = run(XC1 + ["spread_abs"])
r2 = run(XC1 + ["arb_rate"])
r3 = run(XC1 + ["arb_rate", "spread_abs"])

b0 = r0["r2adj"]
print_r("C1 БАЗОВАЯ — без арбитража", r0)
print_r("C1 + spread_abs",            r1, b0)
print_r("C1 + arb_rate",              r2, b0)
print_r("C1 + arb_rate + spread_abs", r3, b0)

# ─── 6. СВОДКА ────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  СВОДНАЯ ТАБЛИЦА")
print(SEP)
print(f"  {'Модель':<28} {'R²adj':>7} {'ΔR²adj':>8} {'F':>8} {'p(F)':>12} "
      f"{'DW':>6} {'JB':>8} {'spread_abs':>12} {'arb_rate':>10}")
print(f"  {'-'*100}")
for lbl, r in [("C1 базовая", r0), ("C1 + spread_abs", r1),
               ("C1 + arb_rate", r2), ("C1 + arb_rate + spread_abs", r3)]:
    delta = f"{r['r2adj']-b0:+.4f}"
    # find arb vars
    sp = arb_p = "—"
    for i, nm in enumerate(r["cols"]):
        if nm == "spread_abs": sp = f"{r['beta'][i]:+.4f} ({sig(r['p'][i])} p={r['p'][i]:.3f})"
        if nm == "arb_rate":   arb_p = f"{r['beta'][i]:+.4f} ({sig(r['p'][i])} p={r['p'][i]:.3f})"
    print(f"  {lbl:<28} {r['r2adj']:>7.4f} {delta:>8} {r['f']:>8.2f} "
          f"{r['fp']:>12.2e} {r['dw']:>6.3f} {r['jb']:>8.1f} {sp:>12}   {arb_p}")

# ─── 7. VIF ───────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  VIF — C1 + spread_abs + arb_rate")
print(SEP)
vif_cols = [c for c in XC1 + ["spread_abs", "arb_rate"] if c in merged.columns]
sv = merged[vif_cols].dropna().replace([np.inf, -np.inf], np.nan).dropna()
for i, col in enumerate(vif_cols):
    try:
        v = variance_inflation_factor(sv.values, i)
        f = "❌ >10" if v > 10 else ("⚠️ 5-10" if v > 5 else "✅ <5")
        print(f"  {col:<22}  VIF = {v:>10.2f}  {f}")
    except Exception:
        print(f"  {col}: ошибка")

# ─── 8. СОХРАНЕНИЕ CSV ────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  ЭКСПОРТ ДАТАСЕТОВ В CSV")
print(SEP)

# CSV 1 — объединённый датасет 90 дней
export_cols = [
    "date",
    # LP метрики
    "LP return daily", "log_LP_full", "fee_APY_pct", "IL_pct",
    "Value of LP position", "LP_Return_fees",
    # Арбитражные переменные
    "n_blocks", "n_arb", "arb_rate", "spread_abs", "spread_pct", "arb_vol",
    # Рыночные переменные
    "ETH_ret1", "Vol_ETH", "Vola_ETH_sq", "ETH_range",
    "Vol_BTC", "BTC_range", "log_Vol_CEX",
    # Пул
    "log_Vol_Pool", "log_TVL", "log_txCount",
    "eth_price_usd", "tvlUSD", "feesUSD", "txCount",
    "Daily_Volume_Pool_ETH_USDC",
    # Позиция и режим
    "ETH_share", "bull", "bear", "LP_lag1",
]
ds = merged[[c for c in export_cols if c in merged.columns]].copy()
ds.to_csv(OUT_DATA, index=False, float_format="%.8f")
print(f"✅ {OUT_DATA}  ({len(ds)} строк × {len(ds.columns)} колонок)")

# CSV 2 — результаты регрессий
rows = []
for lbl, r in [("C1_base", r0), ("C1+spread_abs", r1),
               ("C1+arb_rate", r2), ("C1+both", r3)]:
    meta = dict(Model=lbl, N=r["n"], R2=round(r["r2"],6),
                R2adj=round(r["r2adj"],6), Delta_R2adj=round(r["r2adj"]-b0,6),
                F=round(r["f"],4), F_pvalue=f"{r['fp']:.4e}",
                DW=round(r["dw"],4), JB=round(r["jb"],2),
                JB_pvalue=round(r["jbp"],4),
                Skew=round(r["skew"],4), Kurt=round(r["kurt"],4))
    for i, nm in enumerate(r["cols"]):
        row = {**meta,
               "Variable": nm,
               "Beta": round(r["beta"][i], 8),
               "SE_HC3": round(r["se"][i], 8),
               "t_stat": round(r["t"][i], 4),
               "p_value": round(r["p"][i], 6),
               "Significance": sig(r["p"][i])}
        rows.append(row)

reg_df = pd.DataFrame(rows)
reg_df.to_csv(OUT_REG, index=False)
print(f"✅ {OUT_REG}  ({len(reg_df)} строк — коэффициенты всех моделей)")

print(f"\n{SEP}")
print("  АНАЛИЗ ЗАВЕРШЁН")
print(f"  Вывод: арбитражные переменные (spread_abs, arb_rate) статистически")
print(f"  незначимы на подвыборке 90 дней при новом Vol_ETH (rolling std).")
print(f"  Лучшая модель: C1 базовая (R²adj={b0:.4f}).")
print(SEP)

  ЗАГРУЗКА И АГРЕГАЦИЯ АРБИТРАЖНЫХ ДАННЫХ


KeyError: 'Модуль спреда'